In [ ]:
import pandas as pd
import io

# Only needed if you're uploading fresh in Colab
from google.colab import files
uploaded = files.upload()

Saving marketing_campaign.csv to marketing_campaign.csv


In [ ]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
print(df.shape)
df.head()

(2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [ ]:
# Check how many rows have missing income
print("Missing income values:", df["Income"].isnull().sum())

# Remove rows with missing income
df = df.dropna(subset=["Income"])

# Remove obvious data errors
df = df[df["Income"] < 200000]
df = df[df["Year_Birth"] > 1940]

print("Remaining customers after cleaning:", df.shape[0])

Missing income values: 24
Remaining customers after cleaning: 2211


In [ ]:
# Create Age from birth year
df["Age"] = 2024 - df["Year_Birth"]

# Combine all 6 spending categories into one Total_Spend column
df["Total_Spend"] = (df["MntWines"] + df["MntFruits"] + df["MntMeatProducts"] +
                      df["MntFishProducts"] + df["MntSweetProducts"] + df["MntGoldProds"])

# Combine all purchase channels into one Total_Purchases column
df["Total_Purchases"] = (df["NumDealsPurchases"] + df["NumWebPurchases"] +
                          df["NumCatalogPurchases"] + df["NumStorePurchases"])

# Combine kids and teens into one Children column
df["Children"] = df["Kidhome"] + df["Teenhome"]

# Check that it worked
df[["Age", "Total_Spend", "Total_Purchases", "Children"]].head()

,Age,Total_Spend,Total_Purchases,Children
0,67,1617,25,0
1,70,27,6,2
2,59,776,21,0
3,40,53,8,1
4,43,422,19,1


In [ ]:
from sklearn.preprocessing import StandardScaler

# Select the columns we want to use for grouping customers
features = ["Income", "Age", "Total_Spend", "Total_Purchases", "Children", "NumWebVisitsMonth", "Recency"]
X = df[features].copy()

# Scale everything to the same range
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original income for first customer:", X["Income"].iloc[0])
print("Scaled income for first customer:", X_scaled[0][0])

Original income for first customer: 58138.0
Scaled income for first customer: 0.2870228792462539


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"k={k}: silhouette score = {score:.3f}")

k=2: silhouette score = 0.314
k=3: silhouette score = 0.232
k=4: silhouette score = 0.202
k=5: silhouette score = 0.196
k=6: silhouette score = 0.187
k=7: silhouette score = 0.189
k=8: silhouette score = 0.191


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Segment"] = kmeans.fit_predict(X_scaled)

# See how many customers landed in each segment
df["Segment"].value_counts()

,count
Segment,
3,633
2,619
0,489
1,470


In [ ]:
profile = df.groupby("Segment")[features].mean().round(1)
profile

,Income,Age,Total_Spend,Total_Purchases,Children,NumWebVisitsMonth,Recency
Segment,,,,,,,
0,78484.2,55.0,1389.6,20.1,0.1,2.3,49.9
1,41830.9,60.4,130.1,9.5,1.9,6.0,50.8
2,60877.0,59.5,858.8,22.0,1.1,5.6,47.7
3,30268.0,46.9,112.0,8.0,0.8,6.9,48.3
